# Notebook execution

Run notebook cells and watch Trail record their code, timing and outcome.

Start JupyterLab and Trail as in [setup](setup.md), then open this notebook in `trail-tutorial`. A notebook's kernel is the process that runs its code. Trail listens through Jupyter Server; the normal console workflow needs no extension in the kernel.

Event examples are from earlier runs. Ids, times and notebook or map names will differ.

## Open the notebook

Select the **Python 3** kernel.

Trail records a `KernelSessionEvent` with `verb: attached` when it finds the notebook's kernel. Allow a few seconds for discovery.

```text
100. KernelSessionEvent
    id: 89be196dd3794e3b8aa5d82ea407323c
    timestamp: 12 23:58:39.884
    schema_id: trail/kernel-session/v1
    server: http://localhost:8889/
    src_path: maps/Untitled.ipynb
    verb: attached
    kernel_name: python3
    kernel_id: c6d1baa4-6a55-4ebe-9228-7909604a21d0
```


## Run a cell

In [ ]:
x = 21 * 2


Trail records two `ExecutionEvent`s: `started`, then `completed`.

| Field | Meaning |
|---|---|
| `src_path` | The notebook path. |
| `execution_count` | The number in the cell's execution prompt. |
| `code` | The code that ran. The console shows its first line and the number of remaining lines. |
| `cell_id` | The cell's id, retained when it moves or runs again. |
| `kernel_id` | The kernel that ran it. |
| `duration` | The time from start to finish. |

```text
225. ExecutionEvent
    id: 83563e4a57474405821ff5d10b231fbe
    timestamp: 15:38:43.578
    schema_id: trail/notebook-execution/v1
    server: http://localhost:8899/
    src_path: Untitled6.ipynb
    verb: started
    execution_count: 1
    code: x = 21 * 2
    cell_id: 2ec3ea56-dc68-4915-a02e-f213d499e09b
    kernel_id: b5bdcc4a-4932-4773-835e-8e7a1ab232ba
    entry: fecc6c352c874c8e8948726260ba764e
226. ExecutionEvent
    id: 509316bd1b6345c186b09ef4b4e70b53
    timestamp: 15:38:43.588
    schema_id: trail/notebook-execution/v1
    server: http://localhost:8899/
    src_path: Untitled6.ipynb
    verb: completed
    execution_count: 1
    duration: 0.011 s
    cell_id: 2ec3ea56-dc68-4915-a02e-f213d499e09b
    kernel_id: b5bdcc4a-4932-4773-835e-8e7a1ab232ba
    entry: fecc6c352c874c8e8948726260ba764e
```


## Produce output

In [ ]:
print('hello from a cell')
x


The closing record counts one printed output and one displayed result:

Trail keeps output counts, not output content.

```text
228. ExecutionEvent
    id: 0b4e111e85b64fdbbccca58054d815c2
    timestamp: 15:38:44.721
    schema_id: trail/notebook-execution/v1
    server: http://localhost:8899/
    src_path: Untitled6.ipynb
    verb: completed
    execution_count: 2
    duration: 0.013 s
    outputs: stream 1, execute_result 1
    cell_id: 70ba404d-ee0d-4cd4-9124-b624a56a205e
    kernel_id: b5bdcc4a-4932-4773-835e-8e7a1ab232ba
    entry: fecc6c352c874c8e8948726260ba764e
```


## Cell error

In [ ]:
1 / 0


The closing record has `verb: errored` and reports `ZeroDivisionError: division by zero`. Trail keeps the error name and message, not the traceback.

```text
230. ExecutionEvent
    id: f37cb70887214fd889e15c2593772109
    timestamp: 15:38:45.946
    schema_id: trail/notebook-execution/v1
    server: http://localhost:8899/
    src_path: Untitled6.ipynb
    verb: errored
    execution_count: 3
    duration: 0.094 s
    error: ZeroDivisionError: division by zero
    cell_id: ba3c039d-0daa-4c5f-8531-2e0cd70eb5fb
    kernel_id: b5bdcc4a-4932-4773-835e-8e7a1ab232ba
    entry: fecc6c352c874c8e8948726260ba764e
```


## Restart the kernel

Choose **Kernel → Restart Kernel…**, confirm, then run:

In [ ]:
'after restart'


Trail records `restarted`. The next cell's execution count starts again at 1.

To shut the kernel down, choose **File → Close and Shut Down Notebook**. Trail records it as `detached` from the notebook. Closing a tab alone does not necessarily stop its kernel.

```text
233. KernelSessionEvent
    id: ef4ff016f194498a86cfa9722e758fbb
    timestamp: 15:38:51.033
    schema_id: trail/kernel-session/v1
    server: http://localhost:8899/
    src_path: Untitled6.ipynb
    verb: restarted
    kernel_name: python3
    kernel_id: b5bdcc4a-4932-4773-835e-8e7a1ab232ba
    entry: fecc6c352c874c8e8948726260ba764e
```

## Save the notebook

Press Ctrl+S or let autosave run. Trail groups consecutive saves of the same file into one `ContentsServiceV1Event`, emitted after 30 seconds without another save.

`saves` counts server saves; `writes` counts the associated file writes Trail observed. Those writes are included in the save record rather than repeated as separate file records.

```text
237. ContentsServiceV1Event
    id: 10a1789140c74686900b9ce76e6b41f1
    timestamp: 15:38:53.853
    schema_id: https://events.jupyter.org/jupyter_server/contents_service/v1
    server: http://localhost:8899/
    src_path: Untitled6.ipynb
    started: 15:38:41.508
    saves: 7
    writes: 7
    verb: saved
    entry: fecc6c352c874c8e8948726260ba764e
```

## Recover cells run while Trail was stopped

Reopen the notebook if you shut its kernel down. Enter `quit` in Trail, run a cell, then start `trail` again from `trail-tutorial`.

Trail asks the running kernel for its input history and records unseen cells as `recovered`. It avoids recording the same recovered cell again on later connections.

History contains code and execution numbers, but no execution times, outcomes or cell ids. A recovered record's `paths` lists notebooks using the kernel at connection time; it does not establish which notebook ran the cell. Recovery requires the kernel's history to remain available.

```text
81. ExecutionEvent
    id: a2759c10d2a443bf9215b5bb86a43eee
    timestamp: 15:34:34.435
    schema_id: trail/notebook-execution/v1
    server: http://localhost:8899/
    verb: recovered
    execution_count: 1
    code: from jupytergis import GISDocument (+2 lines)
    paths: ['Untitled1.ipynb']
    kernel_id: 1e6d64f4-bbaa-473d-a832-5cd3c3aa340c
```


## Write a file

Write a tracked GeoJSON copy:

In [ ]:
from pathlib import Path

source = Path('data/crossings.geojson')
target = Path('data/crossings_copy.geojson')
target.write_bytes(source.read_bytes())


The cell produces execution records and file activity. Compare the cell's start and finish with the write's `started` and `timestamp` fields.

Trail does not yet link a write to a cell. Overlapping times are evidence, not proof that the cell caused the write. [File activity](files.ipynb) covers the file records.

## Query notebook history

Run these in Trail:

```text
events cls=ExecutionEvent src_path=notebook.ipynb
events cls=ExecutionEvent verb=errored
events cls=ExecutionEvent verb=recovered
events cls=KernelSessionEvent src_path=notebook.ipynb
events cls=ContentsServiceV1Event src_path=notebook.ipynb
```


## Check file changes

With the Trail console stopped, load its IPython extension in a notebook inside the project:

In [ ]:
%load_ext trail.ipython


The setup environment already includes Trail. If you use another kernel environment, install Trail there first.

After each cell, the extension checks tracked files and records changes as `MetadataEvent`s with `basis: cell`. It leaves recording to the console when one is running. Execution records still require the console.

